# 06 — Baseline Model

**Day 2, Step 6.** Logistic Regression, on purpose: fast, explainable, and it
returns a real probability rather than a label — which matters because Step 16
buckets people into risk bands.

Two upgrades on the Build Notes' snippet:

1. **Repeated stratified K-fold CV** instead of one split. With 237 positives a
   single split is too high-variance to compare models on.
2. **Calibration measured, not assumed.** Risk bands need probabilities that are
   *correct*, not merely correctly *ordered*.

In [1]:
import sys, warnings
sys.path.insert(0, "../src"); sys.path.insert(0, "..")
warnings.filterwarnings("ignore")

import pandas as pd, numpy as np, matplotlib.pyplot as plt
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

# The lab imports from the factory. Nothing below reimplements pipeline logic.
from hrai.utils.config import get, raw_path, seed
from hrai.utils.io import load_raw, load_processed
from hrai.utils.logger import setup_logging
setup_logging(fmt="human")
print(f"seed={seed()}  |  datasets: {sorted(get('datasets'))}")

seed=42  |  datasets: ['employee_attrition', 'essential_skills', 'hr_performance_engagement', 'occupation_data', 'software_skills']


In [2]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_validate, RepeatedStratifiedKFold
from hrai.features.pipeline import build_pipeline
from hrai.ml.evaluate import compute_metrics, reliability_table

df = load_processed("employee_attrition_processed")
y = df["attrition_flag"].astype(int)
X = df.drop(columns=["attrition_flag"])

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=seed())
print(f"train {len(X_tr)}  test {len(X_te)}  positive rate {y_tr.mean():.3f}")

train 1176  test 294  positive rate 0.162


In [3]:
pipe = build_pipeline(X_tr, LogisticRegression(max_iter=2000, class_weight="balanced",
                                              random_state=seed()))
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=seed())
scores = cross_validate(pipe, X_tr, y_tr, cv=cv,
                        scoring=["precision", "recall", "f1", "roc_auc", "average_precision"])
pd.DataFrame({k.replace("test_", ""): [v.mean().round(4), v.std().round(4)]
              for k, v in scores.items() if k.startswith("test_")},
             index=["mean", "std"]).T

2026-08-28 01:54:28 | INFO  | pipeline built


,mean,std
precision,0.3852,0.0395
recall,0.7351,0.0669
f1,0.5035,0.0361
roc_auc,0.8323,0.0263
average_precision,0.6366,0.0503


## Why not accuracy

At a 16.1% positive rate, predicting "stays" for everyone scores 83.9%. Accuracy
rewards exactly the behaviour we are trying to avoid, so it is excluded from the
metric set entirely — not merely de-emphasised.

In [4]:
pipe.fit(X_tr, y_tr)
prob = pipe.predict_proba(X_te)[:, 1]
metrics = compute_metrics(y_te.to_numpy(), prob, 0.5)
pd.Series(metrics.to_dict())

precision             0.3678
recall                0.6809
f1                    0.4776
roc_auc               0.8074
pr_auc                0.5341
brier                 0.1576
threshold             0.5000
true_positives       32.0000
false_positives      55.0000
true_negatives      192.0000
false_negatives      15.0000
support_positive     47.0000
n                   294.0000
dtype: float64

In [5]:
pd.DataFrame(reliability_table(y_te.to_numpy(), prob))

,bucket,n,predicted_rate,observed_rate
0,"(-0.001, 0.1]",70,0.0429,0.0286
1,"(0.1, 0.2]",45,0.1441,0.0222
2,"(0.2, 0.3]",43,0.2480,0.1163
3,"(0.3, 0.4]",31,0.3470,0.1613
4,"(0.4, 0.5]",18,0.4527,0.1111
5,"(0.5, 0.6]",22,0.5459,0.2273
6,"(0.6, 0.7]",22,0.6488,0.2727
7,"(0.7, 0.8]",17,0.7516,0.3529
8,"(0.8, 0.9]",14,0.8621,0.4286
9,"(0.9, 1.0]",12,0.9474,0.7500
